In [ ]:
import sys
if 'google.colab' in sys.modules or True:
    get_ipython().system('pip install -q deep-translator')

import os
import glob
import time
import pandas as pd
import numpy as np
import pydicom
import cv2
import torch
import matplotlib.pyplot as plt
from deep_translator import GoogleTranslator

# -----------------------------------------------------------------------------
# CONFIGURATION CONSTANTS
# -----------------------------------------------------------------------------
COMP = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
TRAIN_SERIES_DIR = f'{COMP}/train_series'
IMAGE_RESOLUTION = 336
K = 32

IM_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IM_STD  = np.array([0.229, 0.224, 0.225], np.float32)

PLANES = {'sagittal': 0, 'coronal': 1, 'axial': 2}
PLANE_NAMES_INV = {v: k for k, v in PLANES.items()}

# -----------------------------------------------------------------------------
# SPATIAL & ORIENTATION HELPER FUNCTIONS
# -----------------------------------------------------------------------------
def plane_from_iop(iop):
    row, col = np.array(iop[:3], float), np.array(iop[3:], float)
    normal_vector = np.cross(row, col)
    plane_idx = int(np.argmax(np.abs(normal_vector)))
    plane_name = ['sagittal', 'coronal', 'axial'][plane_idx]
    return plane_name, row, col

def crop_background(vol_u8):
    mask = vol_u8.max(axis=0) > 8
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return vol_u8
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    y0, x0 = max(0, y0 - 4), max(0, x0 - 4)
    y1, x1 = min(vol_u8.shape[1], y1 + 4), min(vol_u8.shape[2], x1 + 4)
    return vol_u8[:, y0:y1, x0:x1]

def read_series_headers(files):
    heads = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
    iop = next(h.ImageOrientationPatient for h in heads if hasattr(h, 'ImageOrientationPatient'))
    plane, row, col = plane_from_iop(iop)
    return heads, plane, row, col

def compute_slice_ordering(files, heads, plane, row, col):
    nrm = np.cross(row, col)
    pos = np.array([np.dot(np.array(h.ImagePositionPatient, float), nrm) for h in heads])
    order = np.argsort(pos)
    sel = np.linspace(0, len(files) - 1, K).round().astype(int)
    return [order[i] for i in sel]

def load_and_preprocess_pixel_array(files, idx, plane, laterality):
    vol = np.stack([pydicom.dcmread(files[i]).pixel_array for i in idx]).astype(np.float32)
    lo, hi = np.percentile(vol, [0.5, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    vol = (vol * 255).astype(np.uint8)
    vol = crop_background(vol)
    if plane in ('coronal', 'axial') and laterality == 'L':
        vol = vol[:, :, ::-1]
    return vol

def resize_and_normalize_volume(vol, plane):
    out = np.empty((K, IMAGE_RESOLUTION, IMAGE_RESOLUTION), np.float32)
    for j in range(K):
        out[j] = cv2.resize(vol[j], (IMAGE_RESOLUTION, IMAGE_RESOLUTION), interpolation=cv2.INTER_AREA)
    out /= 255.0
    out = out[:, None].repeat(3, axis=1)
    out = (out - IM_MEAN[None, :, None, None]) / IM_STD[None, :, None, None]
    return torch.from_numpy(np.ascontiguousarray(out)), PLANES[plane]

def process_and_display_axial_case(uid, series_uid, confidence_tier, df_train):
    sd = f"{TRAIN_SERIES_DIR}/{uid}/{series_uid}"
    files = sorted(glob.glob(f'{sd}/*.dcm'))
    if not files:
        return
        
    study_row = df_train[df_train['StudyInstanceUID'] == uid].iloc[0]
    
    sample_ds = pydicom.dcmread(files[0])
    metadata = {
        'Modality': getattr(sample_ds, 'Modality', 'MR'),
        'Rows': getattr(sample_ds, 'Rows', None),
        'Columns': getattr(sample_ds, 'Columns', None),
        'PixelSpacing': getattr(sample_ds, 'PixelSpacing', None),
        'SliceThickness': getattr(sample_ds, 'SliceThickness', None),
        'ImageOrientationPatient': getattr(sample_ds, 'ImageOrientationPatient', None),
    }

    heads, plane, row, col = read_series_headers(files)
    idx = compute_slice_ordering(files, heads, plane, row, col)
    vol = load_and_preprocess_pixel_array(files, idx, plane, 'R')
    tensor, plane_id = resize_and_normalize_volume(vol, plane)
    plane_name = PLANE_NAMES_INV.get(plane_id, 'axial')

    raw_report = study_row.get('Report', '')
    translated_report = raw_report
    if pd.notna(raw_report) and str(raw_report).strip():
        try:
            translated_report = GoogleTranslator(source='auto', target='en').translate(str(raw_report))
        except Exception:
            pass

    print(f"\n{'='*70}")
    print(f"TIER: {confidence_tier.upper()} | AXIAL SERIES")
    print(f"{'='*70}")
    print(f"StudyInstanceUID : {uid}")
    print(f"SeriesInstanceUID: {series_uid}")
    print(f"Anatomical Plane : {plane_name.upper()} | Tensor Shape: {tensor.shape}")
    print(f"DICOM Metadata   : {metadata}")
    
    print(f"\n--- REPORT ---")
    print(translated_report if pd.notna(translated_report) and str(translated_report).strip() else "(No Report)")
    print(f"{'-'*70}\n")

    # Plotting slices
    vis_vol = tensor.numpy()
    vis_vol = vis_vol * IM_STD[None, :, None, None] + IM_MEAN[None, :, None, None]
    vis_vol = np.clip(vis_vol, 0, 1)

    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    axes = axes.flatten()
    slice_indices = np.linspace(0, K - 1, 32).astype(int)
    for idx_plot, slice_idx in enumerate(slice_indices):
        img_slice = vis_vol[slice_idx, 0, :, :]
        axes[idx_plot].imshow(img_slice, cmap='gray')
        axes[idx_plot].set_title(f"Slice {slice_idx}", fontsize=8)
        axes[idx_plot].axis('off')

    plt.suptitle(f"[{confidence_tier.upper()}] UID: {uid[:12]}... | Axial Series", fontsize=12)
    plt.tight_layout()
    plt.show()

# -----------------------------------------------------------------------------
# DYNAMIC SELECTION FROM SERIES METADATA (NO HARDCODING)
# -----------------------------------------------------------------------------
df_train = pd.read_csv(f'{COMP}/train.csv')
df_series = pd.read_csv(f'{COMP}/train_series.csv')

print("Filtering series metadata for pure AXIAL views...")

# Filter series dataframe strictly for Axial plane
axial_series_df = df_series[df_series['Anatomical_Plane'].str.lower() == 'axial'].copy()

# Merge with train labels/reports to categorize high vs low confidence based on label extremity / report keywords
merged_df = axial_series_df.merge(df_train, on='StudyInstanceUID', how='inner')

# High confidence criteria: Fracture/abnormality labels clearly 0 or 1, reports with strong definitive phrasing
high_conf_pool = merged_df[merged_df['Fracture'].isin([0, 1])].drop_duplicates(subset=['StudyInstanceUID'])
high_conf_cases = high_conf_pool.head(4)

# Low confidence criteria: Studies with missing explicit fracture tags or ambiguous reports / intermediate label values
low_conf_pool = merged_df[~merged_df['StudyInstanceUID'].isin(high_conf_cases['StudyInstanceUID'])].drop_duplicates(subset=['StudyInstanceUID'])
low_conf_cases = low_conf_pool.head(4)

print(f"Selected {len(high_conf_cases)} High Confidence Axial Cases and {len(low_conf_cases)} Low Confidence Axial Cases dynamically.\n")

print("--- Processing High Confidence AXIAL Cases ---")
for _, row in high_conf_cases.iterrows():
    process_and_display_axial_case(row['StudyInstanceUID'], row['SeriesInstanceUID'], "High Confidence", df_train)

print("\n--- Processing Low Confidence AXIAL Cases ---")
for _, row in low_conf_cases.iterrows():
    process_and_display_axial_case(row['StudyInstanceUID'], row['SeriesInstanceUID'], "Low Confidence", df_train)